In [ ]:
%matplotlib widget

import numpy as np


In [ ]:
rc.animation

In [ ]:
X = np.array([1,2,3]).astype(np.float32)

with open("logtest.spire", 'wb') as f:
    f.write(X.size.to_bytes(4,'big'))
    f.write(X.tobytes())


In [ ]:

f = open("logtest.spire", 'rb')
n_neurons = int.from_bytes(f.read(4), "big")
np.frombuffer(f.read(), dtype=np.float32).reshape(-1, n_neurons)




In [ ]:
import numpy as np
import math
import sys
import bz2
import gzip
import lzma
from pathlib import Path

try:
    from IPython import get_ipython
except ImportError:
    get_ipython = None


def _ipython_shell():
    return get_ipython() if get_ipython is not None else None


def _in_notebook() -> bool:
    shell = _ipython_shell()
    return shell is not None and "IPKernelApp" in shell.config


def _enable_widget_backend() -> bool:
    shell = _ipython_shell()
    if shell is None:
        return False
    try:
        shell.run_line_magic("matplotlib", "widget")
    except Exception as exc:
        print(f"Could not enable widget backend: {exc}")
        return False
    return True


_WIDGET_BACKEND = _enable_widget_backend()

import matplotlib

ffmpeg_path = Path(sys.executable).with_name("ffmpeg")
if ffmpeg_path.exists():
    matplotlib.rcParams["animation.ffmpeg_path"] = str(ffmpeg_path)

matplotlib.rcParams["animation.html"] = "none"

import matplotlib.pyplot as plt
import matplotlib.animation as animation

filename = "./cuda_test_9.spire.gz"
record_stride = 1  # set to the stride used when recording

_SPIRE_ANIMATIONS = []


def _open_spire_file(path: str):
    lower = path.lower()
    if lower.endswith(".spire.gz"):
        return gzip.open(path, "rb")
    if lower.endswith(".spire.xz") or lower.endswith(".spire.lzma"):
        return lzma.open(path, "rb")
    if lower.endswith(".spire.bz2"):
        return bz2.open(path, "rb")
    if lower.endswith(".spire"):
        return open(path, "rb")
    raise ValueError("Unsupported file type.")


def _spire_stem(path: str) -> str:
    name = Path(path).name
    lower = name.lower()
    for ext in (".spire.gz", ".spire.xz", ".spire.lzma", ".spire.bz2", ".spire"):
        if lower.endswith(ext):
            return name[: -len(ext)]
    return Path(path).stem


def _validate_square_recording(neuron_count: int) -> int:
    n = math.isqrt(neuron_count)
    if n * n != neuron_count:
        raise ValueError(f"Recording has {neuron_count} neurons; expected a square count.")
    return n


def play_spire_recording(filename: str, save=False, record_stride: int = 1, fps: int = 60):
    if filename == "":
        return None

    if record_stride < 1:
        raise ValueError("record_stride must be >= 1.")

    try:
        f = _open_spire_file(filename)
    except ValueError:
        print("Cannot parse non-SpiRe file.")
        return None

    try:
        with f:
            header = f.read(4)
            if len(header) < 4:
                print("Invalid or empty recording.")
                return None
            neuron_count = int.from_bytes(header, byteorder="big")
            n = _validate_square_recording(neuron_count)
            tickdata_size = neuron_count * np.dtype(np.float32).itemsize
            frames = []
            while True:
                tick_bytes = f.read(tickdata_size)
                if not tick_bytes:
                    break
                if len(tick_bytes) != tickdata_size:
                    print("Truncated recording.")
                    return None
                frames.append(np.frombuffer(tick_bytes, dtype=np.float32, count=neuron_count))
            if not frames:
                print("Recording contains no frames.")
                return None
            recording = np.vstack(frames)
            print(recording.shape)

    except Exception as e:
        print(e)
        return None

    fig, ax = plt.subplots()
    im = ax.imshow(recording[0].reshape(n, n), cmap="viridis", aspect="auto")
    plt.colorbar(im, ax=ax)
    ax.set_title(f"Tick : {0 * record_stride}")

    def update(frame):
        im.set_array(recording[frame].reshape(n, n))
        ax.set_title(f"Tick : {frame * record_stride}")
        return [im]

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=recording.shape[0],
        interval=1000 / fps,
        blit=True,
        repeat=False,
    )
    _SPIRE_ANIMATIONS.append(anim)

    if save:
        out_path = Path(f"{_spire_stem(filename)}.mp4")
        anim.save(str(out_path), writer="ffmpeg", fps=fps)
        plt.close(fig)
        print(f"Saved {out_path}")
    elif _WIDGET_BACKEND or not _in_notebook():
        plt.show()
    else:
        from IPython.display import Video, display
        out_path = Path(f"{_spire_stem(filename)}.mp4")
        anim.save(str(out_path), writer="ffmpeg", fps=fps)
        plt.close(fig)
        display(Video(str(out_path), embed=False))

    return None


play_spire_recording(filename, save=False, record_stride=record_stride)


In [ ]:
X.size